# Ejercicio 1 - Maximización Cúbica con Algoritmos Genéticos

En este notebook implementa el Ejercicio 1 de la guía de laboratorio: maximizar la función

$$f(x) = x^3 - 4x^2 + 5x$$

El objetivo es a raiz de la clase vista, implementar las funciones, variables y buen manejo de documentación para el desarrollo de esta actividad. La cual esta contemplada por partes, trabajada dentro del repositorio de Github en colaboracion por grupo de trabajo

## Alcance de esta primera entrega

- Definir un espacio de búsqueda acotado donde exista un máximo local claro
- Decodificar cromosomas binarios a valores reales de $x$
- Evaluar la aptitud directamente sobre la función cubica
- Visualizar la evolucion del algoritmo y el dataset de helados como contexto visual
- Dejar la base lista para extender y conectar el Ejercicio 3 sobre la tasa de mutacion

**Dataset:** "https://www.kaggle.com/datasets/mirajdeepbhandari/polynomial-regression"
El dataset consiste en un reporte de temperaturas vs ventas, siendo una relacion no lineal con curvatura clara, perfecto para aplicar la busqueda de coeficientes para AG y aplicacion de la funcion matematica

Ejercicio1 Elaborado por Nicolas Ballesteros

## 1. Configuración e imports

Los pasos iniciales en este ejercicio consisten en crear un entorno virtual venv junto a Jupyter, aplicando python como lenguaje base para compilar el codigo a continuacion. En este ejercicio se usa la version 3.11.15 de Python. Para inicializar, cargamos las librerias principales a utilizar `pandas`, `numpy`, `matplotlib` e `ipywidgets`. Para el dataset, se importa de manera diferente. En entornos locales se invoca dentro de la funcion en el codigo mediante su ruta y nombre del archivo. En caso del entorno de colab en la nube requiere subirse a Google Drive e importar las librerias de google, a la vez que se debe definir la ruta exacta del archivo en cuestion para ser utilizado en el notebook. Dicho esto, a continuacion las librerias a utilizar:

In [ ]:
import copy
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    display = None

random.seed(42)
np.random.seed(42)

plt.style.use("seaborn-v0_8-whitegrid")

print("Imports cargados correctamente.")

Imports cargados correctamente.


## 2. Problema a optimizar

Para este ejercicio se usa un rango acotado donde el maximo local de la funcion cubica quede claro y sea facil de interpretar visualmente. En este notebook se trabajara por defecto con:

- `x_min = 0.0`
- `x_max = 1.55`

Con ese intervalo, el punto crítico en `x = 1` se comporta como el mejor valor de la region explorada

Los valores se pueden ajustar y modificar para obtener variaciones

In [8]:
x_min = 0.0
x_max = 1.55
n_bits = 16
population_size = 40
num_generations = 60
pc = 0.85
pm = 0.03
selection_method = "roulette"
elitism = True


def fitness_cubica(x: float) -> float:
    """calcula la aptitud de la funcion objetivo del ejercicio 1"""
    return x**3 - 4 * x**2 + 5 * x


def decode_binary_to_real(chromosome: list[int], lower_bound: float = x_min, upper_bound: float = x_max) -> float:
    """decodifica un cromosoma binario a un valor real dentro del intervalo definido"""
    integer_value = int("".join(map(str, chromosome)), 2)
    max_integer = (2**len(chromosome)) - 1
    return lower_bound + (integer_value / max_integer) * (upper_bound - lower_bound)


def load_context_dataset() -> pd.DataFrame:
    """Carga el dataset de helados desde la carpeta local del proyecto | Si no existe, devuelve un DataFrame vacío con las columnas esperadas"""
    dataset_path = Path("Dataset") / "Ice_cream selling data.csv"
    if dataset_path.exists():
        return pd.read_csv(dataset_path)
    return pd.DataFrame(columns=["Temperature (°C)", "Ice Cream Sales (units)"])


dataset_helados = load_context_dataset()
dataset_helados.head()

,Temperature (°C),Ice Cream Sales (units)
0,-4.662263,41.842986
1,-4.316559,34.661120
2,-4.213985,39.383001
3,-3.949661,37.539845
4,-3.578554,32.284531


In [ ]:
        
"""selecciona individuos para el cruce | clase y funciones de seleccion para AG"""
class AlgoritmoGenetico:
    """implementacion modular de un algoritmo genetico binario para maximizacion"""


    def __init__(
        self,
        population_size: int,
        chromosome_length: int,
        pc: float,
        pm: float,
        fitness_func,
        decode_func,
        selection_method: str = "roulette",
        elitism: bool = True,
        tournament_size: int = 3,
    ) -> None:
        if not (0 <= pc <= 1 and 0 <= pm <= 1):
            raise ValueError("pc y pm deben estar entre 0 y 1.")
        if population_size <= 0 or chromosome_length <= 0:
            raise ValueError("population_size y chromosome_length deben ser positivos.")
        if selection_method not in {"roulette", "tournament"}:
            raise ValueError("selection_method debe ser 'roulette' o 'tournament'.")

        self.population_size = population_size
        self.chromosome_length = chromosome_length
        self.pc = pc
        self.pm = pm
        self.fitness_func = fitness_func
        self.decode_func = decode_func
        self.selection_method = selection_method
        self.elitism = elitism
        self.tournament_size = tournament_size

        self.population: list[list[int]] = []
        self.max_fitness_history: list[float] = []
        self.avg_fitness_history: list[float] = []
        self.best_individual_genotype: list[int] = []
        self.best_individual_fitness: float = -float("inf")
        self.best_individual_phenotype: float | None = None
        self.best_generation: int | None = None
        self.last_population_snapshots: list[list[list[int]]] = []
        self.last_fitness_snapshots: list[list[float]] = []
        self.last_phenotype_snapshots: list[list[float]] = []
        self.last_history_frame: pd.DataFrame = pd.DataFrame()
        self.last_summary_frame: pd.DataFrame = pd.DataFrame()

    def _initialize_population(self) -> None:
        self.population = [
            [random.randint(0, 1) for _ in range(self.chromosome_length)]
            for _ in range(self.population_size)
        ]

    def _calculate_all_fitness(self, population: list[list[int]]) -> tuple[list[float], list[float]]:
        fitness_values: list[float] = []
        phenotypes: list[float] = []
        for chromosome in population:
            phenotype = self.decode_func(chromosome)
            fitness_value = self.fitness_func(phenotype)
            phenotypes.append(phenotype)
            fitness_values.append(fitness_value)
        return fitness_values, phenotypes

    def _select_proportional(self, population: list[list[int]], fitness_values: list[float]) -> list[list[int]]:
        min_fitness = min(fitness_values)
        adjusted_fitness = [fitness - min_fitness + 1e-6 for fitness in fitness_values] if min_fitness < 0 else fitness_values
        total_fitness = sum(adjusted_fitness)

        if total_fitness == 0:
            return random.choices(population, k=self.population_size)

        return random.choices(population, weights=adjusted_fitness, k=self.population_size)

    def _select_tournament(self, population: list[list[int]], fitness_values: list[float]) -> list[list[int]]:
        selected_population: list[list[int]] = []
        for _ in range(self.population_size):
            contestants = random.sample(range(self.population_size), self.tournament_size)
            winner_index = max(contestants, key=lambda index: fitness_values[index])
            selected_population.append(copy.deepcopy(population[winner_index]))
        return selected_population

    def _crossover_one_point(self, parent1: list[int], parent2: list[int]) -> tuple[list[int], list[int]]:
        if random.random() < self.pc:
            crossover_point = random.randint(1, self.chromosome_length - 1)
            child1 = parent1[:crossover_point] + parent2[crossover_point:]
            child2 = parent2[:crossover_point] + parent1[crossover_point:]
            return child1, child2
        return copy.deepcopy(parent1), copy.deepcopy(parent2)

    def _mutate_flip_bit(self, chromosome: list[int]) -> list[int]:
        mutated = copy.deepcopy(chromosome)
        for index in range(self.chromosome_length):
            if random.random() < self.pm:
                mutated[index] = 1 - mutated[index]
        return mutated

    def _apply_elitism(self, new_population: list[list[int]]) -> list[list[int]]:
        if not self.elitism or not self.best_individual_genotype:
            return new_population

        new_fitness_values, _ = self._calculate_all_fitness(new_population)
        worst_index = int(np.argmin(new_fitness_values))
        new_population[worst_index] = copy.deepcopy(self.best_individual_genotype)
        return new_population

    def run(self, num_generations: int) -> tuple[list[int], float, list[dict]]:
        self._initialize_population()
        self.max_fitness_history = []
        self.avg_fitness_history = []
        self.last_population_snapshots = []
        self.last_fitness_snapshots = []
        self.last_phenotype_snapshots = []
        history_rows: list[dict] = []

        for generation in range(num_generations):
            fitness_values, phenotypes = self._calculate_all_fitness(self.population)
            best_index = int(np.argmax(fitness_values))
            best_fitness = float(fitness_values[best_index])
            best_genotype = copy.deepcopy(self.population[best_index])
            best_phenotype = float(phenotypes[best_index])

            if best_fitness > self.best_individual_fitness:
                self.best_individual_fitness = best_fitness
                self.best_individual_genotype = best_genotype
                self.best_individual_phenotype = best_phenotype
                self.best_generation = generation

            average_fitness = float(np.mean(fitness_values))
            self.max_fitness_history.append(best_fitness)
            self.avg_fitness_history.append(average_fitness)
            self.last_population_snapshots.append(copy.deepcopy(self.population))
            self.last_fitness_snapshots.append(fitness_values.copy())
            self.last_phenotype_snapshots.append(phenotypes.copy())

            history_rows.append(
                {
                    "generacion": generation,
                    "mejor_fitness": best_fitness,
                    "fitness_promedio": average_fitness,
                    "mejor_x": best_phenotype,
                }
            )

            selected_population = (
                self._select_proportional(self.population, fitness_values)
                if self.selection_method == "roulette"
                else self._select_tournament(self.population, fitness_values)
            )

            new_population: list[list[int]] = []
            for index in range(0, self.population_size, 2):
                parent1 = selected_population[index]
                parent2 = selected_population[(index + 1) % self.population_size]
                child1, child2 = self._crossover_one_point(parent1, parent2)
                new_population.append(self._mutate_flip_bit(child1))
                if len(new_population) < self.population_size:
                    new_population.append(self._mutate_flip_bit(child2))

            self.population = self._apply_elitism(new_population[: self.population_size])

        self.last_history_frame = pd.DataFrame(history_rows)
        self.last_summary_frame = pd.DataFrame(
            [
                {
                    "mejor_x": self.best_individual_phenotype,
                    "mejor_fitness": self.best_individual_fitness,
                    "generacion": self.best_generation,
                    "tamano_poblacion": self.population_size,
                    "num_generaciones": num_generations,
                    "pc": self.pc,
                    "pm": self.pm,
                    "elitism": self.elitism,
                    "seleccion": self.selection_method,
                }
            ]
        )

        if self.best_individual_phenotype is not None:
            print(
                "[AG] Finalizado -> "
                f"mejor_x={self.best_individual_phenotype:.6f}, "
                f"fitness={self.best_individual_fitness:.6f}, "
                f"generacion={self.best_generation}"
            )
            print("[AG] Resumen disponible en ag.last_summary_frame y ag.last_history_frame")

        return self.best_individual_genotype, self.best_individual_fitness, history_rows



        """este codigo es para ejecutar el algoritmo genetico y mostrar los resultados en el siguiente bloque de celdas
        Esta seccion del codigo esta inspirada en la actividad y explicacion del ejercicio de introduccion presentado en clase"""

## 3. Ejecucion del algoritmo

Esta celda corre una vez el algoritmo con parametros base y deja un resumen en tabla para inspeccion rapida; en la siguiente seccion se añade la visualizacion interactiva

In [ ]:
ag = AlgoritmoGenetico(
    population_size=population_size,
    chromosome_length=n_bits,
    pc=pc,
    pm=pm,
    fitness_func=fitness_cubica,
    decode_func=decode_binary_to_real,
    selection_method=selection_method,
    elitism=elitism,
)

best_genotype, best_fitness, history_rows = ag.run(num_generations=num_generations)
history_df = pd.DataFrame(history_rows)

best_x = decode_binary_to_real(best_genotype)

print("Mejor cromosoma encontrado:", best_genotype)
print(f"Mejor x encontrado: {best_x:.6f}")
print(f"Mejor fitness encontrado: {best_fitness:.6f}")
print(f"Generación del mejor individuo: {ag.best_generation}")

history_df.head()

Mejor cromosoma encontrado: [1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1]
Mejor x encontrado: 0.999961
Mejor fitness encontrado: 2.000000
Generacion del mejor individuo: 17


,generacion,mejor_fitness,fitness_promedio,mejor_x
0,0,1.999975,1.528249,0.994994
1,1,1.999975,1.873179,0.994994
2,2,1.999975,1.867573,0.994994
3,3,1.999996,1.858425,0.997927
4,4,1.999996,1.878396,0.997927


## 4. Visualizacion interactiva

La siguiente funcion permite revisar una generacion concreta, ver la curva cubica, los individuos de esa generacion y el dataset de helados como referencia visual


In [ ]:
def plot_generation_snapshot(generation_index: int, panel_mode: str = "dataset") -> None:
    generation_index = int(np.clip(generation_index, 0, len(ag.last_population_snapshots) - 1))
    population = ag.last_population_snapshots[generation_index]
    fitness_values = np.asarray(ag.last_fitness_snapshots[generation_index], dtype=float)
    phenotypes = np.asarray(ag.last_phenotype_snapshots[generation_index], dtype=float)

    x_grid = np.linspace(x_min, x_max, 400)
    y_grid = [fitness_cubica(x_value) for x_value in x_grid]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    """Definicion de colores y estilos para las graficas | clasificando por color para caracterizar la funcion objetivo, la poblacion y el mejor individuo global"""

    axes[0].plot(x_grid, y_grid, color="#1f77b4", linewidth=2, label="f(x) = x³ - 4x² + 5x")
    axes[0].scatter(phenotypes, fitness_values, color="#ff7f0e", s=30, alpha=0.8, label="Población")
    axes[0].scatter([best_x], [best_fitness], color="#d62728", s=80, marker="*", label="Mejor global")
    axes[0].axvline(1.0, color="#2ca02c", linestyle="--", linewidth=1.5, label="x = 1")
    axes[0].set_title(f"Generación {generation_index}")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("fitness")
    axes[0].legend()
    

    """Definicion de paneles para mostrar el dataset, el top 5 de la generacion o un resumen de la generacion"""                               
    if panel_mode == "dataset":
        if not dataset_helados.empty:
            temperature_values = dataset_helados["Temperature (°C)"].to_numpy(dtype=float)
            sales_values = dataset_helados["Ice Cream Sales (units)"].to_numpy(dtype=float) 
            axes[1].scatter(
                temperature_values,
                sales_values,
                color="#9467bd",
                alpha=0.75,
                edgecolor="white",
                linewidth=0.5,
                label="Observaciones",
            )
            """temperatura vs ventas de helados con tendencia lineal""" 
            if len(temperature_values) >= 2:
                slope, intercept = np.polyfit(temperature_values, sales_values, 1)
                line_x = np.linspace(temperature_values.min(), temperature_values.max(), 100)
                line_y = slope * line_x + intercept
                axes[1].plot(line_x, line_y, color="#111111", linewidth=2, label="Tendencia lineal")

            axes[1].set_title("Temperatura vs ventas de helados")
            axes[1].set_xlabel("Temperature (°C)")
            axes[1].set_ylabel("Ice Cream Sales (units)")
            axes[1].legend()
        else:
            axes[1].text(0.5, 0.5, "Dataset no disponible", ha="center", va="center", transform=axes[1].transAxes)
            axes[1].set_axis_off()
    elif panel_mode == "top5":
        ranking = np.argsort(fitness_values)[::-1][: min(5, len(fitness_values))]
        top_labels = [f"#{index + 1} x={phenotypes[position]:.4f}" for index, position in enumerate(ranking)]
        top_values = fitness_values[ranking]
        axes[1].barh(top_labels[::-1], top_values[::-1], color="#9467bd")
        axes[1].set_title("Top 5 de la generación")
        axes[1].set_xlabel("fitness")
    else:
        axes[1].axis("off")
        summary_text = (
            f"Mejor fitness: {float(np.max(fitness_values)):.6f}\n"
            f"Fitness promedio: {float(np.mean(fitness_values)):.6f}\n"
            f"Mejor x de la generación: {float(phenotypes[int(np.argmax(fitness_values))]):.6f}\n"
            f"Mejor global: {best_fitness:.6f} en x={best_x:.6f}\n"
            f"Gen. del mejor global: {ag.best_generation}"
        )
        axes[1].text(
            0.05,
            0.95,
            summary_text,
            transform=axes[1].transAxes,
            va="top",
            ha="left",
            fontsize=12,
            bbox={"boxstyle": "round,pad=0.6", "facecolor": "#f5f5f5", "edgecolor": "#999999"},
        )

    plt.tight_layout()
    plt.show()

"""widgets para interactuar con la visualizacion de las generaciones y paneles de informacion"""
if widgets is not None:
    slider = widgets.IntSlider(
        value=len(ag.last_population_snapshots) - 1,
        min=0,
        max=len(ag.last_population_snapshots) - 1,
        step=1,
        description="Generación",
        continuous_update=False,
    )
    panel_mode = widgets.ToggleButtons(
        options=[("Dataset", "dataset"), ("Top 5", "top5"), ("Resumen", "summary")],
        value="dataset",
        description="Panel",
    )
    output = widgets.interactive_output(
        plot_generation_snapshot,
        {"generation_index": slider, "panel_mode": panel_mode},
    )
    display(widgets.VBox([widgets.HBox([slider, panel_mode]), output]))
else:
    plot_generation_snapshot(len(ag.last_population_snapshots) - 1)